# 04 — Bonus Çözümü: Görev Atama

> **Bu dosya `solutions` branch'indedir.**

In [ ]:
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pulp'], check=True)
    for _k in list(sys.modules):
        if _k == 'pulp' or _k.startswith('pulp.'):
            del sys.modules[_k]
    !git clone -q -b solutions https://github.com/Meliksahcaliskan/ILP_Workshop.git
    sys.path.insert(0, "ILP_Workshop")

In [ ]:
import pulp
import pandas as pd
import matplotlib.pyplot as plt
from utils.plotting import show_status, plot_gantt

proc_times = {
    0: {0: 3, 1: 5, 2: 2},
    1: {0: 7, 1: 4, 2: 6},
    2: {0: 2, 1: 3, 2: 8},
    3: {0: 5, 1: 6, 2: 3},
    4: {0: 4, 1: 2, 2: 5},
    5: {0: 6, 1: 7, 2: 4},
    6: {0: 3, 1: 4, 2: 6},
    7: {0: 8, 1: 3, 2: 2},
}
jobs     = list(proc_times.keys())
machines = list(proc_times[0].keys())

## Temel Model — Makespan Minimizasyonu

In [ ]:
prob = pulp.LpProblem("task_assignment", pulp.LpMinimize)

x = {(j, m): pulp.LpVariable(f"x_{j}_{m}", cat="Binary")
     for j in jobs for m in machines}
C_max = pulp.LpVariable("C_max", lowBound=0, cat="Continuous")

prob += C_max
for j in jobs:
    prob += pulp.lpSum(x[j, m] for m in machines) == 1
for m in machines:
    prob += pulp.lpSum(proc_times[j][m] * x[j, m] for j in jobs) <= C_max

prob.solve(pulp.PULP_CBC_CMD(msg=0))
show_status(prob)

cmax_val  = pulp.value(C_max)
assignment = {j: m for j in jobs for m in machines if pulp.value(x[j, m]) > 0.5}

print(f"\nMakespan: {cmax_val:.1f}")
for j in sorted(assignment):
    m = assignment[j]
    print(f"  Görev {j} → İşlemci {m}  (süre: {proc_times[j][m]})")

fig = plot_gantt(assignment, proc_times, num_machines=len(machines), c_max_val=cmax_val)
plt.show()

## Görev 2 — Yasak Atama: Görev 0 İşlemci 2'de Çalışamaz

In [ ]:
prob_g2 = pulp.LpProblem("forbidden", pulp.LpMinimize)
x2 = {(j, m): pulp.LpVariable(f"x_{j}_{m}", cat="Binary")
      for j in jobs for m in machines}
cmax2 = pulp.LpVariable("C_max", lowBound=0)

prob_g2 += cmax2
for j in jobs:
    prob_g2 += pulp.lpSum(x2[j, m] for m in machines) == 1
for m in machines:
    prob_g2 += pulp.lpSum(proc_times[j][m] * x2[j, m] for j in jobs) <= cmax2
prob_g2 += x2[0, 2] == 0  # YASAK

prob_g2.solve(pulp.PULP_CBC_CMD(msg=0))
show_status(prob_g2)
print(f"Yasak atama ile makespan: {pulp.value(cmax2):.1f}")
print(f"Orijinal makespan       : {cmax_val:.1f}")

## Görev 3 — Objective Karşılaştırması: Makespan vs Toplam Süre

In [ ]:
prob_g3 = pulp.LpProblem("total_time", pulp.LpMinimize)
x3 = {(j, m): pulp.LpVariable(f"x_{j}_{m}", cat="Binary")
      for j in jobs for m in machines}

prob_g3 += pulp.lpSum(
    proc_times[j][m] * x3[j, m] for j in jobs for m in machines
)
for j in jobs:
    prob_g3 += pulp.lpSum(x3[j, m] for m in machines) == 1

prob_g3.solve(pulp.PULP_CBC_CMD(msg=0))
show_status(prob_g3)

assign_g3 = {j: m for j in jobs for m in machines if pulp.value(x3[j, m]) > 0.5}
total_v    = int(pulp.value(prob_g3.objective))
makespan_g3 = max(
    sum(proc_times[j][m] for j in jobs if assign_g3.get(j) == m)
    for m in machines
)

print(f"Toplam süre minimize → toplam süre : {total_v}")
print(f"Bu çözümün makespan'i              : {makespan_g3}")
print(f"Orijinal makespan min. makespan    : {int(cmax_val)}")

from utils.plotting import PALETTE
colors_l = [PALETTE["blue"], PALETTE["orange"], PALETTE["green"], PALETTE["sky"],
            PALETTE["vermil"], PALETTE["purple"], PALETTE["black"], PALETTE["gray"]]

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=False)
for ax, (asgn, cv, title) in zip(axes, [
    (assignment, cmax_val,    f"Makespan Minimize  (C_max={cmax_val:.1f})"),
    (assign_g3,  makespan_g3, f"Toplam Süre Minimize  (C_max={makespan_g3})"),
]):
    machine_time = {m_: 0 for m_ in machines}
    for j_ in sorted(asgn):
        m_ = asgn[j_]
        dur = proc_times[j_][m_]
        ax.barh(m_, dur, left=machine_time[m_], color=colors_l[j_ % len(colors_l)],
                edgecolor="white", height=0.55)
        ax.text(machine_time[m_] + dur/2, m_, f"G{j_}",
                ha="center", va="center", fontsize=8, color="white", fontweight="bold")
        machine_time[m_] += dur
    ax.axvline(cv, color=PALETTE["red"], linestyle="--", lw=1.8)
    ax.text(cv + 0.1, len(machines) - 0.7, f"C_max={cv:.1f}",
            color=PALETTE["red"], fontsize=8)
    ax.set_yticks(machines)
    ax.set_yticklabels([f"İşlemci {m_}" for m_ in machines], fontsize=8)
    ax.set_xlabel("Zaman"); ax.set_title(title)
    ax.grid(axis="x", linestyle="--", alpha=0.3)

plt.suptitle("Objective Karşılaştırması: Makespan vs Toplam Süre", fontsize=11)
plt.tight_layout()
plt.show()